In [ ]:
# -- Libraries ----------------------------------------------------------------
import os
import gc
import boto3
import rasterio
import yaml
import numpy as np
from pathlib import Path
from datetime import timedelta
from dotenv import load_dotenv
from pystac_client import Client
from rasterio.windows import Window

# -- Load Configuration -------------------------------------------------------
CONFIG_PATH = Path(__file__).parent / "config.yaml" if "__file__" in dir() else Path("config.yaml")

if not CONFIG_PATH.exists():
    raise FileNotFoundError(
        f"Configuration file not found: {CONFIG_PATH}\n"
        "Please copy 'config.example.yaml' to 'config.yaml' and edit it."
    )

with open(CONFIG_PATH, "r", encoding="utf-8") as f:
    config = yaml.safe_load(f)

# Load environment variables
env_path = Path(".env")
if env_path.exists():
    load_dotenv(env_path)
else:
    print("Warning: .env file not found. S3 credentials may not be available.")
    print("Please copy '.env.example' to '.env' and add your credentials.")

print(f"Libraries loaded  |  rasterio {rasterio.__version__}  |  numpy {np.__version__}")
print(f"Configuration loaded from: {CONFIG_PATH}")

# -- General Settings ---------------------------------------------------------
AOI = config["aoi"]
PROJECT_NAME = config.get("project_name", "NDVI Analysis")

SEARCH_DATE_RANGE  = config["search_date_range"]
TARGET_BANDS       = config.get("target_bands", ["B04_10m", "B08_10m"])
CLOUD_THRESHOLD    = config.get("cloud_threshold", 10.0)
MIN_DAYS_INTERVAL  = config.get("min_days_interval", 14)
MAX_ITEMS          = config.get("max_items", 100)
BLOCK_SIZE         = config.get("block_size", 512)

RAW_DIR       = Path(config["raw_dir"])
PROCESSED_DIR = Path(config["processed_dir"])
RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print(f"\nProject: {PROJECT_NAME}")
print(f"Date range: {SEARCH_DATE_RANGE}")
print(f"Raw data dir      : {RAW_DIR.resolve()}")
print(f"Processed data dir: {PROCESSED_DIR.resolve()}")

Libraries loaded  |  rasterio 1.4.4  |  numpy 2.4.2
Raw data dir     : C:\Users\Sinan\uhuzam-ndvi\data\raw
Processed data dir: C:\Users\Sinan\uhuzam-ndvi\data\processed


In [ ]:
# -- Sentinel-2 Scene Search & Filtering --------------------------------------
catalog = Client.open("https://catalogue.dataspace.copernicus.eu/stac")

search = catalog.search(
    collections=["sentinel-2-l2a"],
    intersects=AOI,
    datetime=SEARCH_DATE_RANGE,
    max_items=MAX_ITEMS,
)

clean_scenes = []
last_date = None

for item in sorted(search.items(), key=lambda x: x.datetime):
    cloud_cover  = item.properties.get("eo:cloud_cover", 100)
    current_date = item.datetime

    if cloud_cover >= CLOUD_THRESHOLD:
        continue
    if last_date is not None and (current_date - last_date) < timedelta(days=MIN_DAYS_INTERVAL):
        continue

    clean_scenes.append(item)
    last_date = current_date
    print(f"{current_date.strftime('%Y-%m-%d')}  |  Cloud: {cloud_cover:.1f}%  |  {item.id}")

print(f"\nTotal scenes selected: {len(clean_scenes)}")

2023-01-02  |  Cloud: 4.0%  |  S2B_MSIL2A_20230102T081239_N0510_R078_T37SDB_20240811T172309
2023-01-22  |  Cloud: 0.1%  |  S2B_MSIL2A_20230122T081139_N0510_R078_T37SDB_20240810T061340
2023-02-11  |  Cloud: 0.5%  |  S2B_MSIL2A_20230211T081009_N0510_R078_T37SDB_20240814T035137
2023-02-26  |  Cloud: 4.8%  |  S2A_MSIL2A_20230226T080911_N0510_R078_T37SDB_20240818T182402
2023-04-17  |  Cloud: 2.4%  |  S2A_MSIL2A_20230417T080611_N0510_R078_T37SDB_20240902T194454
2023-05-02  |  Cloud: 0.3%  |  S2B_MSIL2A_20230502T080609_N0510_R078_T37SDB_20240902T031048
2023-05-27  |  Cloud: 3.2%  |  S2A_MSIL2A_20230527T080611_N0510_R078_T37SDB_20240906T092535
2023-06-11  |  Cloud: 0.1%  |  S2B_MSIL2A_20230611T080609_N0510_R078_T37SDB_20240913T184935
2023-06-26  |  Cloud: 0.5%  |  S2A_MSIL2A_20230626T080611_N0510_R078_T37SDB_20240917T070423
2023-07-11  |  Cloud: 0.0%  |  S2B_MSIL2A_20230711T080609_N0510_R078_T37SDB_20241021T023216
2023-07-26  |  Cloud: 0.0%  |  S2A_MSIL2A_20230726T080611_N0510_R078_T37SDB_2024

In [ ]:
# -- Copernicus S3 Connection -------------------------------------------------
s3 = boto3.client(
    "s3",
    endpoint_url="https://eodata.dataspace.copernicus.eu",
    aws_access_key_id=os.getenv("S3_ACCESS_KEY"),
    aws_secret_access_key=os.getenv("S3_SECRET_KEY"),
    region_name="default",
)

try:
    s3.list_objects_v2(Bucket="eodata", Prefix="Sentinel-2/", Delimiter="/", MaxKeys=1)
    print("S3 connection successful")
except Exception as e:
    print(f"S3 connection error: {e}")
    print("Please check your S3 credentials in the .env file.")

S3 connection successful


In [6]:
# -- Scene Download -----------------------------------------------------------
print(f"Downloading {len(TARGET_BANDS)} bands for {len(clean_scenes)} scenes...\n")

for item in clean_scenes:
    scene_date = item.datetime.strftime("%Y-%m-%d")
    scene_dir  = RAW_DIR / scene_date
    scene_dir.mkdir(parents=True, exist_ok=True)

    print(f"-- {scene_date}")
    for band_name in TARGET_BANDS:
        dest = scene_dir / f"{scene_date}_{band_name}.jp2"

        if dest.exists():
            print(f"   {band_name}: already exists, skipped")
            continue

        s3_key = item.assets[band_name].href.replace("s3://eodata/", "")
        try:
            s3.download_file(Bucket="eodata", Key=s3_key, Filename=str(dest))
            print(f"   {band_name}: done")
        except Exception as e:
            print(f"   {band_name}: error - {e}")

print("\nDownload complete.")



-- 2023-01-02
   B04_10m: already exists, skipped
   B08_10m: already exists, skipped
-- 2023-01-22
   B04_10m: already exists, skipped
   B08_10m: already exists, skipped
-- 2023-02-11
   B04_10m: already exists, skipped
   B08_10m: already exists, skipped
-- 2023-02-26
   B04_10m: already exists, skipped
   B08_10m: already exists, skipped
-- 2023-04-17
   B04_10m: already exists, skipped
   B08_10m: already exists, skipped
-- 2023-05-02
   B04_10m: already exists, skipped
   B08_10m: already exists, skipped
-- 2023-05-27
   B04_10m: already exists, skipped
   B08_10m: already exists, skipped
-- 2023-06-11
   B04_10m: already exists, skipped
   B08_10m: already exists, skipped
-- 2023-06-26
   B04_10m: already exists, skipped
   B08_10m: already exists, skipped
-- 2023-07-11
   B04_10m: already exists, skipped
   B08_10m: already exists, skipped
-- 2023-07-26
   B04_10m: already exists, skipped
   B08_10m: already exists, skipped
-- 2023-08-15
   B04_10m: already exists, skipped
   

In [ ]:
# -- NDVI Computation Function ------------------------------------------------
NODATA = config.get("nodata_value", -9999.0)

def compute_ndvi_windowed(red_path, nir_path, output_path, block_size=512, nodata=-9999.0):
    """
    Compute NDVI from B04 (Red) and B08 (NIR) using windowed reads.
    Output: float32 GeoTIFF, nodata=-9999, NDVI range [-1, 1].
    """
    with rasterio.open(red_path) as red_ds, rasterio.open(nir_path) as nir_ds:
        if red_ds.shape != nir_ds.shape:
            raise ValueError(f"Shape mismatch: B04={red_ds.shape}, B08={nir_ds.shape}")

        meta = red_ds.meta.copy()
        meta.update(driver="GTiff", dtype="float32", count=1, compress="lzw", nodata=nodata)

        with rasterio.open(output_path, "w", **meta) as dst:
            for row in range(0, red_ds.height, block_size):
                win = Window(0, row, red_ds.width, min(block_size, red_ds.height - row))
                red = red_ds.read(1, window=win).astype("float32")
                nir = nir_ds.read(1, window=win).astype("float32")

                denom = nir + red
                ndvi  = np.where(denom > 0, (nir - red) / denom, nodata)

                valid = ndvi != nodata
                ndvi[valid] = np.clip(ndvi[valid], -1.0, 1.0)

                dst.write(ndvi.astype("float32"), 1, window=win)
                del red, nir, denom, ndvi

    gc.collect()

print("compute_ndvi_windowed function ready")

compute_ndvi_windowed function ready


In [ ]:
# -- Batch NDVI Generation ----------------------------------------------------
date_dirs     = sorted([d for d in RAW_DIR.iterdir() if d.is_dir()])
success_count = 0
fail_count    = 0

print(f"{len(date_dirs)} date directories found\n")

for date_dir in date_dirs:
    date_str  = date_dir.name
    red_files = list(date_dir.glob("*B04*"))
    nir_files = list(date_dir.glob("*B08*"))

    if not red_files or not nir_files:
        print(f"  {date_str}: B04 or B08 not found, skipped")
        fail_count += 1
        continue

    output_path = PROCESSED_DIR / f"NDVI_{date_str}.tif"
    try:
        compute_ndvi_windowed(str(red_files[0]), str(nir_files[0]), str(output_path), BLOCK_SIZE, NODATA)
        size_mb = output_path.stat().st_size / (1024 ** 2)
        print(f"  {date_str}  ->  {size_mb:.1f} MB")
        success_count += 1
    except Exception as e:
        print(f"  {date_str}: error - {e}")
        fail_count += 1

    gc.collect()

print(f"\n{'-'*45}")
print(f"Success: {success_count}  |  Failed: {fail_count}")
print(f"Output dir: {PROCESSED_DIR.resolve()}")

🚀 Toplu NDVI üretimi başlatılıyor...

--- 2023-01-02 ---


C:\Users\Sinan\AppData\Local\Temp\ipykernel_7412\850275675.py:51: RuntimeWarning: invalid value encountered in divide
  (nir - red) / denominator,


  ✅ Kaydedildi: NDVI_2023-01-02.tif (513.3 MB)
--- 2023-01-22 ---
  ✅ Kaydedildi: NDVI_2023-01-22.tif (516.5 MB)
--- 2023-02-11 ---
  ✅ Kaydedildi: NDVI_2023-02-11.tif (519.0 MB)
--- 2023-02-26 ---
  ✅ Kaydedildi: NDVI_2023-02-26.tif (520.7 MB)
--- 2023-04-17 ---
  ✅ Kaydedildi: NDVI_2023-04-17.tif (515.6 MB)
--- 2023-05-02 ---
  ✅ Kaydedildi: NDVI_2023-05-02.tif (515.5 MB)
--- 2023-05-27 ---
  ✅ Kaydedildi: NDVI_2023-05-27.tif (512.4 MB)
--- 2023-06-11 ---
  ✅ Kaydedildi: NDVI_2023-06-11.tif (510.3 MB)
--- 2023-06-26 ---
  ✅ Kaydedildi: NDVI_2023-06-26.tif (508.9 MB)
--- 2023-07-11 ---
  ✅ Kaydedildi: NDVI_2023-07-11.tif (509.9 MB)
--- 2023-07-26 ---
  ✅ Kaydedildi: NDVI_2023-07-26.tif (513.0 MB)
--- 2023-08-15 ---
  ✅ Kaydedildi: NDVI_2023-08-15.tif (510.2 MB)
--- 2023-08-30 ---
  ✅ Kaydedildi: NDVI_2023-08-30.tif (512.7 MB)
--- 2023-09-14 ---
  ✅ Kaydedildi: NDVI_2023-09-14.tif (508.1 MB)
--- 2023-09-29 ---
  ✅ Kaydedildi: NDVI_2023-09-29.tif (509.3 MB)
--- 2023-10-14 ---
  ✅ Kayded

In [ ]:
# -- Build Virtual Raster (VRT) -----------------------------------------------
# Stacks all processed NDVI GeoTIFFs into a single multi-band VRT file.
# Each band is labeled with its date. Open the VRT in QGIS and assign any
# three bands to R/G/B channels to visualize temporal change without
# generating large files on disk.

import subprocess

vrt_script = Path(__file__).parent / "build_vrt.py" if "__file__" in dir() else Path("build_vrt.py")

if not vrt_script.exists():
    print(f"Warning: build_vrt.py not found at {vrt_script}")
else:
    result = subprocess.run(
        ["python", str(vrt_script)],
        capture_output=True,
        text=True,
    )
    print(result.stdout)
    if result.returncode != 0:
        print("VRT build failed:")
        print(result.stderr)

Fonksiyonlar hazır ✅


In [ ]:
# -- RGB Time Series Composite (Optional) -------------------------------------
# Run this cell only if you want physical .tif files for each composite group.
# Maps 3 consecutive NDVI files to R-G-B bands to visualize temporal change.
#
# WARNING: Each composite can be several hundred MB depending on the study area.
#          Check available disk space before running.
#
# If you only need QGIS visualization, use the VRT built in the previous cell
# instead — it achieves the same result without writing large files to disk.

def normalize_ndvi(band, nodata=-9999.0):
    """Normalize NDVI [-1, 1] to [0, 1]. Nodata pixels are set to 0."""
    out = np.zeros_like(band, dtype="float32")
    v   = band != nodata
    out[v] = (band[v] + 1.0) / 2.0
    return out

def create_rgb_composite(files, output_path, nodata=-9999.0):
    """Write 3 NDVI files as R/G/B bands into a single GeoTIFF."""
    with rasterio.open(files[0]) as ref:
        meta = ref.meta.copy()
        h, w = ref.height, ref.width

    meta.update(driver="GTiff", dtype="float32", count=3, compress="lzw")
    meta.pop("nodata", None)

    with rasterio.open(output_path, "w", **meta) as dst:
        for i in range(3):
            if i < len(files):
                with rasterio.open(files[i]) as src:
                    dst.write(normalize_ndvi(src.read(1), nodata), i + 1)
            else:
                dst.write(np.zeros((h, w), dtype="float32"), i + 1)
    gc.collect()


# -- Run ----------------------------------------------------------------------
ndvi_files = sorted(PROCESSED_DIR.glob("NDVI_*.tif"))

if len(ndvi_files) < 3:
    print(f"RGB composite requires at least 3 NDVI files (found: {len(ndvi_files)})")
else:
    # Sliding window with step=2: 0-1-2, 2-3-4, 4-5-6 ...
    groups = [ndvi_files[i:i+3] for i in range(0, len(ndvi_files) - 2, 2)]
    print(f"Creating {len(groups)} RGB composites...\n")

    for i, group in enumerate(groups):
        dates = [f.stem.replace("NDVI_", "") for f in group]
        label = f"{dates[0]}_to_{dates[-1]}"
        out   = PROCESSED_DIR / f"RGB_NDVI_{i+1:02d}_{label}.tif"

        print(f"Group {i+1}: R={dates[0]}  G={dates[1]}  B={dates[2]}")
        try:
            create_rgb_composite(group, str(out), NODATA)
            print(f"  {out.name}  ({out.stat().st_size/(1024**2):.1f} MB)")
        except Exception as e:
            print(f"  Error: {e}")
        gc.collect()

    print("\nDone.")

In [ ]:
# -- Data Summary -------------------------------------------------------------
ndvi_files = sorted(PROCESSED_DIR.glob("NDVI_*.tif"))

if not ndvi_files:
    print("No NDVI files found")
else:
    print(f"NDVI Summary for {PROJECT_NAME}  ({len(ndvi_files)} files)\n{'-'*50}")
    with rasterio.open(ndvi_files[0]) as src:
        print(f"CRS            : {src.crs}")
        print(f"Resolution (m) : {src.res}")
        print(f"Shape          : {src.height} x {src.width}")
        print(f"Bounds         : {src.bounds}")

        valid = src.read(1)
        valid = valid[valid != NODATA]
        if len(valid):
            print(f"NDVI range     : {valid.min():.3f} - {valid.max():.3f}")
            print(f"Mean NDVI      : {valid.mean():.3f}")

    dates = [f.stem.replace("NDVI_", "") for f in ndvi_files]
    print(f"\nDate range: {dates[0]}  ->  {dates[-1]}")

CRS: EPSG:32637
Resolution (m): (10.0, 10.0)
Shape (rows, cols): (10980, 10980)
Transform:
| 10.00, 0.00, 399960.00|
| 0.00,-10.00, 4200000.00|
| 0.00, 0.00, 1.00|
Bounds: BoundingBox(left=399960.0, bottom=4090200.0, right=509760.0, top=4200000.0)
Data type: uint16

Pixel value range: 0 - 18720
Array shape: (10980, 10980)
